In [1]:
#|default_exp lawa

In [2]:
#| hide
import nbdev; nbdev.nbdev_export()

/usr/local/lib/python3.12/dist-packages/nbdev/export.py:88: UserWarning: Notebook '/workspaces/gpt/rugptxl_converter.ipynb' uses `#|export` without `#|default_exp` cell.
Note nbdev2 no longer supports nbdev1 syntax. Run `nbdev_migrate` to upgrade.
See https://nbdev.fast.ai/getting_started.html for more information.
  warn(f"Notebook '{nbname}' uses `#|export` without `#|default_exp` cell.\n"


In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "3"
os.environ["VLLM_USE_V1"] = "0"
os.environ["VLLM_ATTENTION_BACKEND"] = "FLASHINFER"
#os.environ['VLLM_LOGGING_LEVEL'] = 'DEBUG'

In [4]:
#| export
from os import getenv
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer,AutoConfig
from front.common import process_seq
import random
from rest.gen import get_line_enders, iftoken
model_path = getenv("MODEL")
gpu_part = int(getenv("gpu_fraction",20))/100

INFO 04-28 01:52:22 [__init__.py:239] Automatically detected platform cuda.


In [5]:
gpu_part

0.2

In [ ]:
# Step 0: loss: 3.4908 lr (3e-5)/2 final Loss: 3.1622 8bit eval: error vllm bfloat16: 3.1763
model_path = 'large/poetry'

# valid Loss: 2.987608 8bit eval: 2.9492 vllm bfloat16: 2.9595
model_path = 'xl/poetry'

# Step 0: 5.5830 lr (3e-5)/2 final Loss: 2.6281 8bit eval: 2.6289 vllm bfloat16: 2.6286 TGI 16/8bit: 2.6279 
model_path = 'mig'

# Step 0: loss: 2.9164 lr (3e-5)/2 final Loss: 2.7707 8bit eval: error vllm bfloat16: 2.7712
model_path = 'large/pelevin'

# Step 0: 2.7811 lr 3e-6 final Loss: 2.5810 8bit eval: 2.5998 vllm bfloat16: 2.5873 TGI 16bit: 2.5881 TGI 8bit: 2.5908
model_path = 'lawa'

# Step 0: 4.2097 lr (3e-5)/2 final Loss: 2.6410; 8bit eval: 2.6445 vllm bfloat16: 2.6413 TGI: 2.6422
model_path = 'xl/pelevin'

# Step 0: 3.2267 lr 3e-4 final Loss: 0.7983
model_path = 'charly1.3'

# Step 0: 1.0042 lr 3e-4 final Loss: 0.7276
model_path = 'charly1.3'

# Step 0: 2.5511 lr = 3e-5 Loss: 2.4204 bpbs: 0.637344
model_path = 'tuned_models/rugpt13B'


In [7]:
#| export
full_path = f'./models/{model_path}'
tokenizer = AutoTokenizer.from_pretrained(full_path)

# saving VRAM
config = AutoConfig.from_pretrained(full_path)
model_max_length = config.max_position_embeddings
max_model_len = min(4096, model_max_length)

model = LLM(model=full_path, dtype="bfloat16", device="cuda", gpu_memory_utilization=gpu_part, max_model_len=max_model_len
            , enable_chunked_prefill=True, max_num_batched_tokens=4,max_num_seqs=4,
            # ── additional VRAM savers ────────────────────────────────────────────
            kv_cache_dtype="fp8",     # 2× smaller KV cache (needs Hopper/Ada+CUDA 11.8) :contentReference[oaicite:1]{index=1}
            calculate_kv_scales=True, # on-the-fly scales if you didn't pre-calibrate
            swap_space=8,            # GiB of CPU RAM to spill rare long contexts)
)

INFO 04-28 01:52:30 [config.py:689] This model supports multiple tasks: {'generate', 'classify', 'score', 'embed', 'reward'}. Defaulting to 'generate'.
INFO 04-28 01:52:30 [config.py:1316] Using fp8 data type to store kv cache. It reduces the GPU memory footprint and boosts the performance. Meanwhile, it may cause accuracy drop without a proper scaling factor
INFO 04-28 01:52:30 [config.py:1901] Chunked prefill is enabled with max_num_batched_tokens=4.
INFO 04-28 01:52:30 [llm_engine.py:243] Initializing a V0 LLM engine (v0.8.4) with config: model='./models/xl/pelevin', speculative_config=None, tokenizer='./models/xl/pelevin', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=2048, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dty

[W428 01:52:32.581333296 socket.cpp:759] [c10d] The client socket cannot be initialized to connect to [bbb]:47689 (errno: 97 - Address family not supported by protocol).


Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]


INFO 04-28 01:52:35 [loader.py:458] Loading weights took 2.90 seconds
INFO 04-28 01:52:35 [model_runner.py:1146] Model loading took 2.4510 GiB and 2.948290 seconds
INFO 04-28 01:52:36 [worker.py:267] Memory profiling takes 0.49 seconds
INFO 04-28 01:52:36 [worker.py:267] the current vLLM instance can use total_gpu_memory (23.54GiB) x gpu_memory_utilization (0.20) = 4.71GiB
INFO 04-28 01:52:36 [worker.py:267] model weights take 2.45GiB; non_torch_memory takes 0.08GiB; PyTorch activation peak memory takes 0.35GiB; the rest of the memory reserved for KV Cache is 1.83GiB.
INFO 04-28 01:52:36 [executor_base.py:112] # cuda blocks: 1252, # CPU blocks: 5461
INFO 04-28 01:52:36 [executor_base.py:117] Maximum concurrency for 2048 tokens per request: 9.78x
INFO 04-28 01:52:39 [model_runner.py:1456] Capturing cudagraphs for decoding. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI. If 

Capturing CUDA graph shapes:   0%|          | 0/3 [00:00<?, ?it/s]

WARNING 04-28 01:52:39 [config.py:3934] Current vLLM config is not set.
WARNING 04-28 01:52:40 [config.py:3934] Current vLLM config is not set.
WARNING 04-28 01:52:41 [config.py:3934] Current vLLM config is not set.
INFO 04-28 01:52:41 [model_runner.py:1598] Graph capturing finished in 2 secs, took 0.08 GiB
INFO 04-28 01:52:41 [llm_engine.py:449] init engine (profile, create kv cache, warmup model) took 6.21 seconds


In [8]:
model.llm_engine.model_config.max_model_len

2048

In [ ]:
#| export
stop_token_ids = iftoken(tokenizer, ['<|endoftext|>','|eot_id|','<|end_of_text|>','<|eot_id|>','<pad>'])
line_enders = get_line_enders(tokenizer)

def create_logit_bias(tokenizer, blocked_tokens, penalty: float = float('-inf')):
    """Return {token_id: bias} mapping to ‘hard-ban’ tokens in V1."""
    blocked_token_ids = iftoken(tokenizer, blocked_tokens)
    return {tid: penalty for tid in blocked_token_ids}

def create_token_blocker(tokenizer, blocked_tokens):
    blocked_token_ids = iftoken(tokenizer, blocked_tokens)
    
    def token_blocker(input_ids, scores):
        if scores.dim() == 2:
            scores[:, list(blocked_token_ids)] = -float('inf')
        elif scores.dim() == 1:
            scores[list(blocked_token_ids)] = -float('inf')
        else:
            raise ValueError(f"Unexpected score tensor shape: {scores.shape}")
        return scores
    
    return token_blocker

def get_sampling_params(tokenizer, length: int, num_samples: int, allow_linebreak: bool, temperature: float):
    blocked_tokens = ['?»',"».",'http://',',[','("','.]',' («',')','\u2004',']','(«','[', ' [', '(', ' (', '\xa0', '*', '­', '~', '_', '\\', '\uf04a', '\ufeff', '\u2028']
    if not allow_linebreak:
        blocked_tokens.extend(line_enders)
    
    logit_bias = create_logit_bias(tokenizer, blocked_tokens)
    token_blocker = create_token_blocker(tokenizer, blocked_tokens)

    return SamplingParams(
        max_tokens=length,
        n=num_samples,
        min_p=0.1,
        stop_token_ids=stop_token_ids,
        ignore_eos=True,
        logits_processors=[token_blocker],
        #logit_bias=logit_bias,
        repetition_penalty=2.,
        temperature=temperature,
        seed=random.randint(0, 1000000),
    )

In [10]:
stop_token_ids

[1]

In [11]:
#| export
def get_sample(prompt: str, length: int, num_samples: int, allow_linebreak: bool, temperature: float = 1.0):
    max_input = model.llm_engine.model_config.max_model_len - length
    prompt = tokenizer.decode(tokenizer.encode(prompt)[-max_input:]).removeprefix('<|begin_of_text|>')
    sampling_params = get_sampling_params(tokenizer, length, num_samples, allow_linebreak, temperature)
    outputs = model.generate(prompt, sampling_params)
    generated_sequences = [oo.text for o in outputs for oo in o.outputs]
    return process_seq(generated_sequences)

In [12]:
%%time
get_sample('На словах ты Лев Толстой, а на деле'*100000, 300, 4, False)

Processed prompts:   0%|          | 0/4 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

CPU times: user 21.9 s, sys: 497 ms, total: 22.4 s
Wall time: 22.4 s


['',
 ' На языках тебе надо говорить: "Лев толстой!"А он отвечает:" А я не могу иначе..." Вот это и есть графомания. Она начинается с того момента когда человек начинает писать о себе в третьем лице."Я" - главное слово во всем романе". Это цитата из предисловия к роману самого Аксенова-Чхартишвили.) В России еще можно встретить человека пишущему об Америке или Англии от первого лица; чаще всего такие люди страдают манией величия либо просто хотят казаться умнее других по принципу «лучше быть богатым дураком…» Но вот что пишет сам автор романа про себя уже несколько лет как вышедший за пределы этих представлений…« Я» пишется всегда только через дефис.» Отношение автора ко многим вещам сильно зависит оттого насколько они интересны читателю – читательский интерес проявляется даже тогда если вещь написана человеком совершенно неинтересным для него лично»,– считает писатель Владимир Маканин.[ 1 ] По его собственному признанию эта особенность написания многих романов привела впоследствии пис

In [13]:
%%time
get_sample('На словах ты Лев Толстой, а на деле', 300, 4, False)

Processed prompts:   0%|          | 0/4 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

CPU times: user 1.97 s, sys: 3.66 ms, total: 1.97 s
Wall time: 1.97 s


[' говно. Учи матчасть… Так что не ссы – прорвемся!» Он был прав и в другом: мы действительно пробили дно еще глубже; теперь уже нас было сложно остановить или повернуть назад к цивилизации хотя бы по той причине все наши мысли были направлены исключительно внутрь собственного тела с его загадочными таинственными процессами внутри клеток мозга за секунду до того момента как эти процессы вступали во взаимодействие друг со мной самим из-за своего полного несоответствия моему представлению о том самом главном механизме человеческой жизни…» Но почему же тогда я ничего этого раньше сам себе объяснить так хорошо мне понятного языка у меня никак ни разу даже толком само это понятие выскочить наружу? Я ведь вроде старался понять суть происходящего вокруг себя самого! А она оказалась простой донельзя настолько проста для понимания любого мало грамотного человека среднего возраста после нескольких стаканов водки без закуски под «Колбасу» да пары сигарет при выключенном свете от фонаря напротив п